In [0]:

from pyspark.sql import functions as F

catalog = "dbw_healthcare_de_dev"
gold_table = f"{catalog}.gold.DimPatientCurrent"

dbutils.widgets.text("file_id", "")
dbutils.widgets.text("adf_run_id", "")

file_id = dbutils.widgets.get("file_id").strip()
adf_run_id = dbutils.widgets.get("adf_run_id").strip()

if not file_id or not adf_run_id:
    raise ValueError("file_id and adf_run_id are required")

# Validate ownership and Gold processing stage
control = spark.sql("""
SELECT batch_id
FROM dbw_healthcare_de_dev.etl.file_control
WHERE file_id = :file_id
  AND adf_run_id = :adf_run_id
  AND processing_stage = 'GOLD'
  AND pipeline_status = 'IN_PROGRESS'
  AND is_enabled = TRUE
""", args={
    "file_id": file_id,
    "adf_run_id": adf_run_id
}).collect()

if len(control) != 1:
    raise RuntimeError(
        "Gold processing rejected: invalid ownership or stage"
    )

batch_id = control[0]["batch_id"]

# Verify at least one successful Silver attempt
silver_audit = spark.sql("""
SELECT 1
FROM dbw_healthcare_de_dev.etl.file_stage_audit
WHERE file_id = :file_id
  AND processing_stage = 'SILVER'
  AND stage_status = 'SUCCESS'
  AND stage_end_ts IS NOT NULL
LIMIT 1
""", args={"file_id": file_id}).collect()

if not silver_audit:
    raise RuntimeError(
        "Gold processing requires a completed Silver audit"
    )

# Create schema and Delta table automatically
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog}.gold
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_table} (
    DimPatientID BIGINT,
    PatientId STRING,
    FirstName STRING,
    LastName STRING,
    Gender STRING,
    DateOfBirth DATE,
    State STRING,
    BatchId STRING,
    RowStartDate TIMESTAMP,
    LastUpdatedTimestamp TIMESTAMP
)
USING DELTA
""")

print("Gold schema and table ready")
print("Gold table:", gold_table)
print("File ID:", file_id)
print("Batch ID:", batch_id)


# Create Gold initialization control table
gold_control_table = f"{catalog}.etl.gold_load_control"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_control_table} (
    table_name STRING,
    initial_load_status STRING,
    initial_load_completed_ts TIMESTAMP,
    last_updated_ts TIMESTAMP
)
USING DELTA
""")

print("Gold initialization control table ready")


In [0]:

# Validate active Silver patient records

silver_table = f"{catalog}.silver.DimPatients"

df_silver = spark.table(silver_table)

# Validate entire Silver dimension before filtering
invalid_active_counts = (
    df_silver
    .groupBy("PatientId")
    .agg(
        F.sum(
            F.when(F.col("IsActive") == True, 1)
             .otherwise(0)
        ).alias("active_count")
    )
    .filter(F.col("active_count") != 1)
)

if invalid_active_counts.limit(1).count() > 0:
    display(invalid_active_counts)
    raise ValueError(
        "Silver validation failed: every patient must "
        "have exactly one active record"
    )

# Extract current patient records
df_gold_source = df_silver.filter(
    F.col("IsActive") == True
)

# Validate non-empty dataset
silver_active_count = df_gold_source.count()

if silver_active_count == 0:
    raise ValueError(
        "Gold load rejected: no active patients found"
    )

# Validate patient identifiers
invalid_patient_ids = df_gold_source.filter(
    F.col("PatientId").isNull() |
    (F.trim(F.col("PatientId")) == "")
)

if invalid_patient_ids.limit(1).count() > 0:
    raise ValueError(
        "Gold load rejected: null or blank PatientId"
    )

# Check duplicates
duplicate_patients = (
    df_gold_source
    .groupBy("PatientId")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_patients.limit(1).count() > 0:
    raise ValueError(
        "Gold load rejected: duplicate active PatientId"
    )

print("Silver validation passed")
print("Active patients:", silver_active_count)

# Identify patients affected by the current file
bronze_table = f"{catalog}.bronze.patients_raw"

df_changed_ids = (
    spark.table(bronze_table)
    .filter(
        (F.col("file_id") == file_id) &
        (F.col("batch_id") == batch_id)
    )
    .select(F.col("patient_id").alias("PatientId"))
    .distinct()
)

changed_patient_count = df_changed_ids.count()

if changed_patient_count == 0:
    raise ValueError(
        "Gold load rejected: no patients found for this file"
    )

# Prepare incremental source from current Silver versions
df_incremental_source = (
    df_gold_source
    .join(df_changed_ids, on="PatientId", how="inner")
)

if df_incremental_source.count() != changed_patient_count:
    raise ValueError(
        "Gold load rejected: changed patients do not "
        "match current Silver records"
    )

print("Changed patients:", changed_patient_count)
print("Gold incremental source prepared")


In [0]:

# Determine Gold load mode

control_rows = spark.sql(f"""
SELECT
    table_name,
    initial_load_status
FROM {gold_control_table}
WHERE table_name = :table_name
""", args={
    "table_name": gold_table
}).collect()

# Validate control table integrity
if len(control_rows) > 1:
    raise RuntimeError(
        "Duplicate Gold initialization control records"
    )

if not control_rows:
    gold_load_mode = "BACKFILL"

elif control_rows[0]["initial_load_status"] == "COMPLETED":
    gold_load_mode = "INCREMENTAL"

elif control_rows[0]["initial_load_status"] in (
    "NOT_STARTED", "IN_PROGRESS"
):
    raise RuntimeError(
        "Gold initialization is incomplete. "
        "Reconciliation required before proceeding."
    )

else:
    raise RuntimeError(
        "Invalid Gold initialization status"
    )

print("Gold load mode:", gold_load_mode)


In [0]:

# Validate Gold initialization state

if gold_load_mode == "BACKFILL":

    # Ensure no prior Gold backfill audit exists
    previous_gold_attempts = spark.sql("""
        SELECT COUNT(*) AS attempt_count
        FROM dbw_healthcare_de_dev.etl.file_stage_audit
        WHERE processing_stage = 'GOLD'
    """).collect()[0]["attempt_count"]

    if previous_gold_attempts > 0:
        raise RuntimeError(
            "Previous Gold attempts exist. "
            "Reconcile initialization before backfill."
        )

    # Confirm Gold table is empty before first backfill
    existing_gold_count = spark.table(gold_table).limit(1).count()

    if existing_gold_count > 0:
        raise RuntimeError(
            "Gold table already contains records. "
            "Initial backfill requires reconciliation."
        )

    print("Initial Gold backfill prechecks passed")

else:
    # Incremental processing requires a completed backfill
    print("Gold backfill completed previously")
    print("Incremental MERGE permitted")


In [0]:

# GOLD INITIALIZATION — Record backfill start

if gold_load_mode == "BACKFILL":

    # Recheck that no initialization record exists
    existing_control = spark.sql(f"""
        SELECT table_name
        FROM {gold_control_table}
        WHERE table_name = :table_name
    """, args={
        "table_name": gold_table
    }).collect()

    if existing_control:
        raise RuntimeError(
            "Gold initialization already has a control record. "
            "Reconciliation required."
        )

    # Record initialization as IN_PROGRESS
    spark.sql(f"""
        INSERT INTO {gold_control_table} (
            table_name,
            initial_load_status,
            initial_load_completed_ts,
            last_updated_ts
        )
        VALUES (
            :table_name,
            'IN_PROGRESS',
            NULL,
            CURRENT_TIMESTAMP()
        )
    """, args={
        "table_name": gold_table
    })

    # Verify initialization record
    init_rows = spark.sql(f"""
        SELECT initial_load_status
        FROM {gold_control_table}
        WHERE table_name = :table_name
    """, args={
        "table_name": gold_table
    }).collect()

    if (
        len(init_rows) != 1
        or init_rows[0]["initial_load_status"] != "IN_PROGRESS"
    ):
        raise RuntimeError(
            "Gold initialization claim verification failed"
        )

    print("Gold initialization marked IN_PROGRESS")

else:
    print("Gold initialization already completed")
    print("Incremental processing permitted")


In [0]:

# Gold BACKFILL / INCREMENTAL Delta MERGE

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# --------------------------------------------------
# 1. Define Gold columns
# --------------------------------------------------

gold_columns = [
    "DimPatientID",
    "PatientId",
    "FirstName",
    "LastName",
    "Gender",
    "DateOfBirth",
    "State",
    "BatchId",
    "RowStartDate"
]

# --------------------------------------------------
# 2. Select source based on Gold load mode
# --------------------------------------------------

if gold_load_mode == "BACKFILL":

    # First load: all current active Silver patients
    df_source = df_gold_source.select(*gold_columns)

elif gold_load_mode == "INCREMENTAL":

    # Later loads: patients affected by this file
    df_source = df_incremental_source.select(*gold_columns)

else:
    raise ValueError(
        f"Unsupported Gold load mode: {gold_load_mode}"
    )

source_count = df_source.count()

if source_count == 0:
    raise ValueError(
        "Gold MERGE rejected: source contains no patients"
    )

# --------------------------------------------------
# 3. Validate source and target schemas
# --------------------------------------------------

gold_schema = spark.table(gold_table).schema

source_types = {
    field.name: field.dataType
    for field in df_source.schema.fields
}

target_types = {
    field.name: field.dataType
    for field in gold_schema.fields
}

missing_target_columns = (
    set(gold_columns) - set(target_types)
)

if missing_target_columns:
    raise ValueError(
        f"Missing Gold columns: {sorted(missing_target_columns)}"
    )

type_mismatches = {
    name: (
        str(source_types[name]),
        str(target_types[name])
    )
    for name in gold_columns
    if source_types[name] != target_types[name]
}

if type_mismatches:
    raise ValueError(
        f"Gold/Silver datatype mismatch: {type_mismatches}"
    )

# --------------------------------------------------
# 4. Add Gold refresh timestamp
# --------------------------------------------------

df_source = df_source.withColumn(
    "LastUpdatedTimestamp",
    F.current_timestamp()
)

# --------------------------------------------------
# 5. Initialize Delta table reference
# --------------------------------------------------

gold_delta = DeltaTable.forName(
    spark,
    gold_table
)
# GOLD AUDIT — START
import uuid

gold_audit_id = str(uuid.uuid4())

spark.sql("""
CALL dbw_healthcare_de_dev.etl.usp_manage_stage_audit(
    'START',
    :audit_id,
    :file_id,
    :adf_run_id,
    'GOLD',
    NULL,
    NULL,
    NULL,
    NULL
)
""", args={
    "audit_id": gold_audit_id,
    "file_id": file_id,
    "adf_run_id": adf_run_id
})

# AUDIT START 
# Verify that the audit was actually created
audit_rows = spark.sql("""
SELECT audit_id, attempt_number, stage_status
FROM dbw_healthcare_de_dev.etl.file_stage_audit
WHERE audit_id = :audit_id
  AND file_id = :file_id
  AND adf_run_id = :adf_run_id
  AND processing_stage = 'GOLD'
  AND stage_status = 'IN_PROGRESS'
  AND stage_end_ts IS NULL
""", args={
    "audit_id": gold_audit_id,
    "file_id": file_id,
    "adf_run_id": adf_run_id
}).collect()

if len(audit_rows) != 1:
    raise RuntimeError(
        "Gold audit START failed or another attempt is in progress"
    )

print("Gold audit started successfully")
print("Audit ID:", gold_audit_id)
print("Attempt:", audit_rows[0]["attempt_number"])

# --------------------------------------------------
# 6. Perform Delta MERGE
# --------------------------------------------------

(
    gold_delta.alias("target")
    .merge(
        df_source.alias("source"),
        "target.PatientId = source.PatientId"
    )
    .whenMatchedUpdate(
        condition="""
            NOT (
                target.DimPatientID <=> source.DimPatientID
                AND target.FirstName <=> source.FirstName
                AND target.LastName <=> source.LastName
                AND target.Gender <=> source.Gender
                AND target.DateOfBirth <=> source.DateOfBirth
                AND target.State <=> source.State
                AND target.BatchId <=> source.BatchId
                AND target.RowStartDate <=> source.RowStartDate
            )
        """,
        set={
            "DimPatientID": "source.DimPatientID",
            "FirstName": "source.FirstName",
            "LastName": "source.LastName",
            "Gender": "source.Gender",
            "DateOfBirth": "source.DateOfBirth",
            "State": "source.State",
            "BatchId": "source.BatchId",
            "RowStartDate": "source.RowStartDate",
            "LastUpdatedTimestamp":
                "source.LastUpdatedTimestamp"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

# --------------------------------------------------
# 7. Log results
# --------------------------------------------------

print("Gold Delta MERGE completed successfully")
print("Gold table:", gold_table)
print("Load mode:", gold_load_mode)
print("Source patient count:", source_count)
print("File ID:", file_id)
print("Batch ID:", batch_id)


In [0]:

# Gold post-MERGE validation

# Select the expected records for this load
expected = df_source.select(*gold_columns)

# Read the corresponding patients from Gold
source_patient_ids = expected.select("PatientId")

actual = (
    spark.table(gold_table)
    .join(source_patient_ids, on="PatientId", how="inner")
    .select(*gold_columns)
)

# 1. Validate processed patient counts
expected_count = expected.count()
actual_count = actual.count()

if actual_count != expected_count:
    raise ValueError(
        f"Gold count mismatch: expected={expected_count}, "
        f"actual={actual_count}"
    )

# 2. Validate no duplicate PatientId values
duplicate_gold = (
    actual.groupBy("PatientId")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_gold.limit(1).count() > 0:
    raise ValueError(
        "Gold validation failed: duplicate PatientId"
    )

# 3. Compare patient attributes
missing_in_gold = expected.exceptAll(actual).limit(1).count()
unexpected_in_gold = actual.exceptAll(expected).limit(1).count()

if missing_in_gold > 0 or unexpected_in_gold > 0:
    raise ValueError(
        "Gold validation failed: patient attributes differ from Silver"
    )

# 4. During initial backfill, validate total population
if gold_load_mode == "BACKFILL":
    total_gold_count = spark.table(gold_table).count()

    if total_gold_count != silver_active_count:
        raise ValueError(
            f"Gold backfill mismatch: Silver={silver_active_count}, "
            f"Gold={total_gold_count}"
        )

# Store counts for the audit completion cell
gold_count = actual_count

print("Gold validation PASSED")
print("Load mode:", gold_load_mode)
print("Expected patients:", expected_count)
print("Validated Gold patients:", gold_count)


In [0]:

# Complete Gold audit and file processing

# 1. Confirm Gold audit is still in progress
audit_check = spark.sql("""
SELECT audit_id
FROM dbw_healthcare_de_dev.etl.file_stage_audit
WHERE audit_id = :audit_id
  AND file_id = :file_id
  AND adf_run_id = :adf_run_id
  AND processing_stage = 'GOLD'
  AND stage_status = 'IN_PROGRESS'
  AND stage_end_ts IS NULL
""", args={
    "audit_id": gold_audit_id,
    "file_id": file_id,
    "adf_run_id": adf_run_id
}).collect()

if len(audit_check) != 1:
    raise RuntimeError(
        "Gold completion rejected: active audit not found"
    )

# 2. Mark Gold audit successful
spark.sql("""
CALL dbw_healthcare_de_dev.etl.usp_manage_stage_audit(
    'SUCCESS',
    :audit_id,
    :file_id,
    :adf_run_id,
    'GOLD',
    :source_count,
    :processed_count,
    0,
    NULL
)
""", args={
    "audit_id": gold_audit_id,
    "file_id": file_id,
    "adf_run_id": adf_run_id,
    "source_count": source_count,
    "processed_count": gold_count
})

# 3. Verify audit SUCCESS
success_check = spark.sql("""
SELECT audit_id
FROM dbw_healthcare_de_dev.etl.file_stage_audit
WHERE audit_id = :audit_id
  AND stage_status = 'SUCCESS'
  AND stage_end_ts IS NOT NULL
""", args={"audit_id": gold_audit_id}).collect()

if len(success_check) != 1:
    raise RuntimeError("Gold audit SUCCESS verification failed")

# 4 Record successful initial Gold backfill

if gold_load_mode == "BACKFILL":

    spark.sql(f"""
    MERGE INTO {gold_control_table} AS target
    USING (
        SELECT
            :table_name AS table_name,
            'COMPLETED' AS initial_load_status,
            CURRENT_TIMESTAMP() AS initial_load_completed_ts,
            CURRENT_TIMESTAMP() AS last_updated_ts
    ) AS source
    ON target.table_name = source.table_name

    WHEN MATCHED THEN UPDATE SET
        target.initial_load_status = source.initial_load_status,
        target.initial_load_completed_ts =
            source.initial_load_completed_ts,
        target.last_updated_ts = source.last_updated_ts

    WHEN NOT MATCHED THEN INSERT (
        table_name,
        initial_load_status,
        initial_load_completed_ts,
        last_updated_ts
    )
    VALUES (
        source.table_name,
        source.initial_load_status,
        source.initial_load_completed_ts,
        source.last_updated_ts
    )
    """, args={"table_name": gold_table})

    print("Gold initial backfill marked COMPLETED")

else:
    print("Incremental Gold load completed")


# 5 Verify Gold initialization status after BACKFILL

if gold_load_mode == "BACKFILL":

    init_check = spark.sql(f"""
        SELECT initial_load_status
        FROM {gold_control_table}
        WHERE table_name = :table_name
    """, args={
        "table_name": gold_table
    }).collect()

    if (
        len(init_check) != 1
        or init_check[0]["initial_load_status"] != "COMPLETED"
    ):
        raise RuntimeError(
            "Gold initialization verification failed"
        )

    print("Gold initialization status verified: COMPLETED")



# 6. Advance file_control to COMPLETED
spark.sql("""
CALL dbw_healthcare_de_dev.etl.usp_update_file_stage(
    :file_id,
    :adf_run_id,
    'GOLD'
)
""", args={
    "file_id": file_id,
    "adf_run_id": adf_run_id
})

# 7. Verify final file status
final_status = spark.sql("""
SELECT processing_stage, pipeline_status
FROM dbw_healthcare_de_dev.etl.file_control
WHERE file_id = :file_id
  AND adf_run_id = :adf_run_id
""", args={
    "file_id": file_id,
    "adf_run_id": adf_run_id
}).collect()

if (
    len(final_status) != 1
    or final_status[0]["processing_stage"] != "COMPLETED"
    or final_status[0]["pipeline_status"] != "COMPLETED"
):
    raise RuntimeError(
        "Gold processing completed but file_control "
        "was not updated correctly"
    )

print("Gold audit SUCCESS")
print("File processing COMPLETED")
print("File ID:", file_id)
print("Gold patient count:", gold_count)



